# Indic Parler-TTS Test
This notebook tests the Indic Parler-TTS model.

In [ ]:
!pip install --upgrade protobuf transformers accelerate soundfile scipy
!pip install git+https://github.com/huggingface/parler-tts.git

In [ ]:
# Uninstall conflicting versions
!pip uninstall -y protobuf tensorflow google-cloud-aiplatform

# Install exact versions
!pip install --no-cache-dir "protobuf>=4.25.0,<5.0.0" transformers accelerate soundfile scipy
!pip install git+https://github.com/huggingface/parler-tts.git

# Restart runtime may be needed here

In [ ]:
import torch
from huggingface_hub import login
from transformers import AutoTokenizer
from parler_tts import ParlerTTSForConditionalGeneration
import soundfile as sf

# --- 1. Authenticate ---
my_token = "YOUR_HF_TOKEN_HERE"
login(token=my_token)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}\n")

# --- 2. Load Model ---
print("Loading Indic Parler-TTS model...")
parler_id = "ai4bharat/indic-parler-tts"

text_tokenizer = AutoTokenizer.from_pretrained(parler_id, use_fast=False)
desc_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
parler_model = ParlerTTSForConditionalGeneration.from_pretrained(parler_id).to(device)

# --- 3. Generate ---
text_to_speak = "हैलो, मेरा नाम जय है।"
voice_description = "A male speaker with a clear, natural voice talking at a calm, steady pace in a quiet room."

desc_tokens = desc_tokenizer(voice_description, return_tensors="pt")
text_tokens = text_tokenizer(text_to_speak, return_tensors="pt")

print("Generating audio...")
with torch.no_grad():
    generation = parler_model.generate(
        input_ids=desc_tokens.input_ids.to(device),
        attention_mask=desc_tokens.attention_mask.to(device),
        prompt_input_ids=text_tokens.input_ids.to(device),
        prompt_attention_mask=text_tokens.attention_mask.to(device),
    )

audio_arr_p = generation.cpu().numpy().squeeze()
sf.write("indic_parler_output.wav", audio_arr_p, parler_model.config.sampling_rate)
print("✅ Saved to indic_parler_output.wav")